# Getting started

pyRADDOSE-3D calculates the radiation dose a crystal absorbs during a
diffraction experiment, voxel by voxel and image by image, as RADDOSE-3D
does. This notebook describes an experiment, runs it, reads the results and
writes RADDOSE-3D's output files.

## Describing an experiment

An experiment is a crystal, a beam, and one or more wedges of rotation.
They are usually written down in a YAML file. This is the example from
`programs/dose`, the insulin crystal from RADDOSE-3D's own example input:

In [ ]:
from pathlib import Path

from pyraddose_3d.models.experiment import Experiment

DOSE = Path("../../programs/dose")
print((DOSE / "dose_example_config.yaml").read_text())

In [ ]:
experiment = Experiment.from_file(DOSE / "dose_example_config.yaml")
experiment.beam

An existing RADDOSE-3D input file is read just as well -- `from_file` reads
anything not ending in `.yaml` or `.yml` as one. `dose_example_input.txt`
is RADDOSE-3D's `MyInput.txt`, unmodified, and it describes the same
experiment:

In [ ]:
Experiment.from_file(DOSE / "dose_example_input.txt") == experiment

Every setting is checked when it is read, so a mistake is an error at
load time rather than a wrong dose later. A setting RADDOSE-3D requires is
required here, and one it defaults is defaulted to the same value:

In [ ]:
from pydantic import ValidationError

try:
    Experiment.model_validate(
        {**experiment.model_dump(), "beam": {"type": "gaussian", "flux": -2e12}}
    )
except ValidationError as error:
    print(error)

## Running it

`solve` returns one summary per wedge: the sixteen numbers RADDOSE-3D
reports.

In [ ]:
from pyraddose_3d.solve import solve

for summary in solve(experiment):
    print(
        f"wedge {summary.wedge_number}: "
        f"average DWD {float(summary.average_dwd):6.2f} MGy, "
        f"max dose {float(summary.max_dose):6.2f} MGy, "
        f"efficiency {float(summary.diffraction_efficiency):.3g} photons/MGy"
    )

`simulate` returns everything behind those numbers: the voxel grid, the
dose in every voxel, and -- with `record_images=True` -- the dose and
fluence of every voxel after every image. From it, RADDOSE-3D's printed
summary:

In [ ]:
from pyraddose_3d.outputs import summary_text
from pyraddose_3d.solve import simulate

simulation = simulate(experiment, record_images=True)
print(summary_text(simulation))

## Writing RADDOSE-3D's files

`write_outputs` writes the files RADDOSE-3D writes by default, under the
same names and in the same formats, so anything that reads RADDOSE-3D's
output reads these unchanged.

In [ ]:
import tempfile

from pyraddose_3d.outputs import write_outputs

directory = Path(tempfile.mkdtemp())
for path in write_outputs(simulation, prefix=str(directory / "output-")):
    print(f"{path.name:28s} {path.stat().st_size:>9,d} bytes")

## A first look at the dose

The final dose of every voxel inside the crystal, as `DoseState.csv` lists
it. The beam is narrower than the crystal, so most of it gets very little:

In [ ]:
import matplotlib.pyplot as plt

from pyraddose_3d.outputs import final_dose_state

state = final_dose_state(simulation)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(state["dose"], bins=50)
ax.set_xlabel("dose after both wedges (MGy)")
ax.set_ylabel("voxels")
ax.set_yscale("log")
plt.show()